In [4]:
import os
import time

import numpy as np
import onnxruntime as ort
import tensorflow as tf
import tf2onnx
from PIL import Image


# ==========================================================
# Paths
# ==========================================================

KERAS_PATH = "Model_EfficientNetV2B0/brain_tumor_model-v-4.keras"
ONNX_PATH = "Model_EfficientNetV2B0/brain_tumor_model.onnx"

In [2]:
# ==========================================================
# Load Test Dataset
# ==========================================================

test_ds = tf.keras.utils.image_dataset_from_directory(
    "Datatset/Testing",
    validation_split=0.5,
    subset="validation",
    seed=42,
    image_size=(256, 256),
    batch_size=64,
    color_mode="grayscale",
    label_mode="categorical",
    shuffle=False,
)

images_batch, labels_batch = next(iter(test_ds))

images_np = images_batch.numpy()
labels_np = np.argmax(labels_batch.numpy(), axis=1)

class_names = test_ds.class_names


Found 1600 files belonging to 4 classes.
Using 800 files for validation.


In [5]:
# ==========================================================
# Load Keras Model
# ==========================================================

model = tf.keras.models.load_model(KERAS_PATH)
model.summary()


# ==========================================================
# Convert Keras → ONNX
# ==========================================================

input_signature = (
    tf.TensorSpec(
        shape=(None, 256, 256, 1),
        dtype=tf.float32,
        name="mri_input",
    ),
)

model_proto, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=input_signature,
    opset=17,
    output_path=ONNX_PATH,
)

size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)

print(f"\n✅ ONNX model saved → {ONNX_PATH}")
print(f"Model size: {size_mb:.1f} MB")


f:\my\Course\RoadMap_Ai_LLM_VLM\projects\MRI_classification\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 255 variables whereas the saved optimizer has 508 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "Brain_Tumor_EfficientNetV2B0"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256, 256,  │          0 │ input_layer_2[0]… │
│ (Concatenate)       │ 3)                │            │ input_layer_2[0]… │
│                     │                   │            │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Rescaling           │ (None, 256, 256,  │          0 │ concatenate[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetv2-b0   │ (None, 8, 8,      │  5,919,312 │ Rescaling[0][0]   │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ efficientnetv2-b… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 1280)      │      5,120 │ global_average_p… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1280)      │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    327,936 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     16,512 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 4)         │        516 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,542,186 (47.84 MB)

 Trainable params: 6,239,380 (23.80 MB)

 Non-trainable params: 63,424 (247.75 KB)

 Optimizer params: 6,239,382 (23.80 MB)




✅ ONNX model saved → Model_EfficientNetV2B0/brain_tumor_model.onnx
Model size: 23.9 MB


In [6]:
# ==========================================================
# Create ONNX Runtime Session
# ==========================================================

session = ort.InferenceSession(
    ONNX_PATH,
    providers=[
        "CUDAExecutionProvider",
        "CPUExecutionProvider",
    ],
)

inp = session.get_inputs()[0]
outp = session.get_outputs()[0]

print("\n✅ Session Ready")
print(f"Input      : {inp.name} | {inp.shape} | {inp.type}")
print(f"Output     : {outp.name} | {outp.shape} | {outp.type}")
print(f"Device      : {ort.get_device()}")
print(f"Providers   : {session.get_providers()}")


f:\my\Course\RoadMap_Ai_LLM_VLM\projects\MRI_classification\.venv\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(



✅ Session Ready
Input      : mri_input | ['unk__1402', 256, 256, 1] | tensor(float)
Output     : dense_3 | ['unk__1403', 4] | tensor(float)
Device      : CPU
Providers   : ['CPUExecutionProvider']


In [7]:
# ==========================================================
# Sanity Check
# ==========================================================

dummy = np.random.randint(
    0,
    256,
    (1, 256, 256, 1),
).astype(np.float32)

dummy_out = session.run(
    [outp.name],
    {inp.name: dummy},
)[0]

print(
    f"\nSanity Check Passed "
    f"→ Output Shape: {dummy_out.shape}, "
    f"Sum: {dummy_out.sum():.4f}"
)


Sanity Check Passed → Output Shape: (1, 4), Sum: 1.0000


In [8]:
# ==========================================================
# Prediction Function
# ==========================================================

def predict_mri(
    image_source,
    session,
    class_names,
    img_size=(256, 256),
    top_k=None,
):
    # ------------------------------------------------------
    # Load Image
    # ------------------------------------------------------

    if isinstance(image_source, str):

        img = (
            Image.open(image_source)
            .convert("L")
            .resize(img_size)
        )

        img_array = np.array(img, dtype=np.float32)

    elif isinstance(image_source, np.ndarray):

        img = Image.fromarray(image_source.astype(np.uint8))

        if img.mode != "L":
            img = img.convert("L")

        img = img.resize(img_size)
        img_array = np.array(img, dtype=np.float32)

    elif isinstance(image_source, Image.Image):

        img = (
            image_source
            .convert("L")
            .resize(img_size)
        )

        img_array = np.array(img, dtype=np.float32)

    else:

        raise TypeError(
            f"image_source باید str، ndarray یا PIL.Image باشد، نه {type(image_source)}"
        )

    img_array = img_array[np.newaxis, :, :, np.newaxis]
    # ------------------------------------------------------
    # Inference
    # ------------------------------------------------------

    inp_name = session.get_inputs()[0].name
    out_name = session.get_outputs()[0].name

    probs = session.run(
        [out_name],
        {inp_name: img_array},
    )[0][0]

    predicted_idx = int(np.argmax(probs))

    # ------------------------------------------------------
    # Probabilities
    # ------------------------------------------------------

    prob_dict = {
        name: float(probs[i])
        for i, name in enumerate(class_names)
    }

    if top_k is not None:

        top_indices = np.argsort(probs)[::-1][:top_k]

        prob_dict = {
            class_names[i]: float(probs[i])
            for i in top_indices
        }

    return {
        "predicted_class": class_names[predicted_idx],
        "confidence": float(probs[predicted_idx]),
        "probabilities": prob_dict,
    }

In [10]:
# ==========================================================
# Single Image Prediction
# ==========================================================

result = predict_mri(
    "Datatset/Testing/meningioma/Te-aug-me_7.jpg",
    session,
    class_names,
    top_k=4,
)

print("\nPredicted Class :", result["predicted_class"])
print(f"Confidence      : {result['confidence'] * 100:.2f}%")

print("\nClass Probabilities")

for cls, prob in result["probabilities"].items():

    bar = "█" * int(prob * 30)

    print(
        f"{cls:<15}"
        f"{prob * 100:6.2f}%   {bar}"
    )



Predicted Class : meningioma
Confidence      : 99.52%

Class Probabilities
meningioma      99.52%   █████████████████████████████
glioma           0.30%   
notumor          0.11%   
pituitary        0.07%   


In [11]:
# ==========================================================
# Keras Inference
# ==========================================================

start = time.time()

keras_preds = model.predict(
    images_batch,
    verbose=0,
)

keras_time = time.time() - start
keras_cls = np.argmax(keras_preds, axis=1)


# ==========================================================
# ONNX Inference
# ==========================================================

inp_name = session.get_inputs()[0].name
out_name = session.get_outputs()[0].name

start = time.time()

onnx_preds = session.run(
    [out_name],
    {inp_name: images_np},
)[0]

onnx_time = time.time() - start
onnx_cls = np.argmax(onnx_preds, axis=1)


In [12]:
# ==========================================================
# Evaluation
# ==========================================================

keras_acc = (keras_cls == labels_np).mean()
onnx_acc = (onnx_cls == labels_np).mean()
agreement = (keras_cls == onnx_cls).mean()

print("\n==================== Results ====================")
print(f"Keras Accuracy : {keras_acc * 100:.2f}%")
print(f"ONNX Accuracy  : {onnx_acc * 100:.2f}%")
print(f"Keras Time     : {keras_time * 1000:.1f} ms")
print(f"ONNX Time      : {onnx_time * 1000:.1f} ms")
print(f"Agreement      : {agreement * 100:.2f}%")
print("=================================================")


==================== Results ====================
Keras Accuracy : 98.44%
ONNX Accuracy  : 98.44%
Keras Time     : 14889.5 ms
ONNX Time      : 2063.0 ms
Agreement      : 100.00%
